# W1 Day2 (04/08 周二): Python 元编程 + 设计模式

## 学习目标
- **理论 (1-1.5h)**: 装饰器三层 / 描述符 / 元类; 常用设计模式 (工厂 / 策略 / 观察者)
- **实践 (2h)**: @timer @retry @cache 装饰器库 + 元类注册表 + 设计模式实现
- **产出物**: 装饰器库 + 设计模式实现

---

## 目录
1. [装饰器: 从基础到三层结构](#1)
2. [实战装饰器库: @timer / @retry / @cache / @validate](#2)
3. [描述符协议 (Descriptor Protocol)](#3)
4. [元类 (Metaclass)](#4)
5. [元类实战: 自动注册表](#5)
6. [设计模式: 工厂模式](#6)
7. [设计模式: 策略模式](#7)
8. [设计模式: 观察者模式](#8)
9. [综合实战: ML 训练框架中的设计模式](#9)
10. [思考题 & 总结](#10)

In [ ]:
import time
import functools
import inspect
import logging
import random
from abc import ABC, abstractmethod
from typing import Any, Callable, Dict, List, Optional, Type
from collections import defaultdict
from dataclasses import dataclass, field

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)

---
## 1. 装饰器: 从基础到三层结构 <a id='1'></a>

### 1.1 装饰器的本质

装饰器就是一个**接收函数并返回函数**的高阶函数。

```python
@decorator
def func():
    pass

# 等价于:
func = decorator(func)
```

### 1.2 三层结构

| 层级 | 结构 | 用途 |
|------|------|------|
| 无参装饰器 | `decorator(func)` → `wrapper` | 最简单 |
| 带参装饰器 | `decorator(args)` → `decorator(func)` → `wrapper` | 参数化行为 |
| 可选参数 | 同时支持 `@deco` 和 `@deco(args)` | 最灵活 |

In [ ]:
# 1.1 最简单的装饰器
print("=" * 60)
print("装饰器基础")
print("=" * 60)

def simple_logger(func):
    """无参装饰器: 记录函数调用"""
    @functools.wraps(func)  # 保留原函数的 __name__, __doc__ 等
    def wrapper(*args, **kwargs):
        print(f"  → 调用 {func.__name__}({args}, {kwargs})")
        result = func(*args, **kwargs)
        print(f"  ← {func.__name__} 返回 {result}")
        return result
    return wrapper

@simple_logger
def add(a, b):
    """两数相加"""
    return a + b

result = add(3, 5)
print(f"\n函数名保留: {add.__name__} (没有 @wraps 会变成 'wrapper')")
print(f"文档保留:   {add.__doc__}")

In [ ]:
# 1.2 带参数的装饰器 (三层嵌套)
print("\n" + "=" * 60)
print("带参数的装饰器 (三层结构)")
print("=" * 60)

def repeat(n_times=2):
    """带参装饰器: 重复执行函数 n 次"""
    def decorator(func):           # 第二层: 接收被装饰的函数
        @functools.wraps(func)
        def wrapper(*args, **kwargs):  # 第三层: 实际的 wrapper
            results = []
            for i in range(n_times):
                result = func(*args, **kwargs)
                results.append(result)
            return results
        return wrapper
    return decorator              # 第一层: 返回真正的装饰器

@repeat(n_times=3)
def greet(name):
    msg = f"Hello, {name}! (time={time.time():.4f})"
    print(f"  {msg}")
    return msg

print("@repeat(n_times=3):")
results = greet("World")
print(f"  返回 {len(results)} 个结果")

In [ ]:
# 1.3 可选参数装饰器 (同时支持 @deco 和 @deco(args))
print("\n" + "=" * 60)
print("可选参数装饰器")
print("=" * 60)

def debug(func=None, *, prefix="DEBUG"):
    """
    可选参数装饰器:
      @debug          → func 不是 None
      @debug()        → func 是 None, prefix 用默认值
      @debug(prefix=) → func 是 None, prefix 用自定义值
    """
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            print(f"  [{prefix}] {fn.__name__} called")
            return fn(*args, **kwargs)
        return wrapper
    
    if func is not None:
        # 用法: @debug (无括号)
        return decorator(func)
    else:
        # 用法: @debug() 或 @debug(prefix="...")
        return decorator

@debug
def func_a():
    return "a"

@debug(prefix="TRACE")
def func_b():
    return "b"

print("@debug (无括号):")
func_a()
print("\n@debug(prefix='TRACE'):")
func_b()

In [ ]:
# 1.4 类装饰器 (装饰类) + 装饰器类 (用类实现装饰器)
print("\n" + "=" * 60)
print("类装饰器 & 装饰器类")
print("=" * 60)

# 装饰器类: 用 __call__ 实现
class CountCalls:
    """用类实现的装饰器: 统计函数被调用次数"""
    def __init__(self, func):
        functools.update_wrapper(self, func)
        self.func = func
        self.count = 0
    
    def __call__(self, *args, **kwargs):
        self.count += 1
        result = self.func(*args, **kwargs)
        return result

@CountCalls
def say_hello():
    return "hello"

for _ in range(5):
    say_hello()
print(f"say_hello 被调用了 {say_hello.count} 次")

# 类装饰器: 装饰一个类
def singleton(cls):
    """单例装饰器: 确保类只有一个实例"""
    instances = {}
    @functools.wraps(cls)
    def get_instance(*args, **kwargs):
        if cls not in instances:
            instances[cls] = cls(*args, **kwargs)
        return instances[cls]
    return get_instance

@singleton
class Database:
    def __init__(self):
        self.id = id(self)
        print(f"  Database 创建 (id={self.id})")

print("\n单例模式:")
db1 = Database()
db2 = Database()
print(f"  db1 is db2: {db1 is db2}  ← 同一个实例")

---
## 2. 实战装饰器库 <a id='2'></a>

构建一套可复用的装饰器工具库。

In [ ]:
# ===================== @timer =====================
def timer(func=None, *, unit='s', log_args=False):
    """
    计时装饰器
    
    用法:
        @timer
        @timer(unit='ms')
        @timer(unit='ms', log_args=True)
    """
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            start = time.perf_counter()
            result = fn(*args, **kwargs)
            elapsed = time.perf_counter() - start
            
            if unit == 'ms':
                elapsed_str = f"{elapsed * 1000:.2f}ms"
            elif unit == 'us':
                elapsed_str = f"{elapsed * 1e6:.2f}μs"
            else:
                elapsed_str = f"{elapsed:.4f}s"
            
            args_str = f" args={args}, kwargs={kwargs}" if log_args else ""
            print(f"  ⏱ {fn.__name__}{args_str} → {elapsed_str}")
            return result
        return wrapper
    
    if func is not None:
        return decorator(func)
    return decorator


# 测试
print("=" * 60)
print("@timer 装饰器")
print("=" * 60)

@timer
def slow_func():
    time.sleep(0.1)
    return 42

@timer(unit='ms', log_args=True)
def compute(n):
    return sum(i**2 for i in range(n))

slow_func()
compute(100000)

In [ ]:
# ===================== @retry =====================
def retry(func=None, *, max_attempts=3, delay=1.0, backoff=2.0, 
          exceptions=(Exception,)):
    """
    重试装饰器 (指数退避)
    
    Args:
        max_attempts: 最大重试次数
        delay: 初始延迟 (秒)
        backoff: 退避倍数
        exceptions: 需要重试的异常类型
    """
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            current_delay = delay
            last_exception = None
            
            for attempt in range(1, max_attempts + 1):
                try:
                    return fn(*args, **kwargs)
                except exceptions as e:
                    last_exception = e
                    if attempt < max_attempts:
                        print(f"  ⚠ {fn.__name__} 第{attempt}次失败: {e}, "
                              f"{current_delay:.1f}s 后重试...")
                        time.sleep(current_delay)
                        current_delay *= backoff
                    else:
                        print(f"  ✗ {fn.__name__} {max_attempts}次全部失败")
            
            raise last_exception
        return wrapper
    
    if func is not None:
        return decorator(func)
    return decorator


# 测试
print("\n" + "=" * 60)
print("@retry 装饰器")
print("=" * 60)

call_count = 0

@retry(max_attempts=4, delay=0.1, backoff=2.0, exceptions=(ConnectionError,))
def flaky_api_call():
    global call_count
    call_count += 1
    if call_count < 3:  # 前2次失败
        raise ConnectionError(f"Connection refused (attempt {call_count})")
    return f"Success on attempt {call_count}"

call_count = 0
result = flaky_api_call()
print(f"  结果: {result}")

In [ ]:
# ===================== @cache (手写 LRU Cache) =====================
from collections import OrderedDict

def lru_cache(maxsize=128):
    """
    手写 LRU Cache 装饰器
    (理解 functools.lru_cache 的原理)
    
    使用 OrderedDict: 最近使用的移到末尾，满了删头部
    """
    def decorator(fn):
        cache = OrderedDict()
        hits = 0
        misses = 0
        
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            nonlocal hits, misses
            
            # 生成缓存 key (args + sorted kwargs)
            key = (args, tuple(sorted(kwargs.items())))
            
            if key in cache:
                hits += 1
                cache.move_to_end(key)  # 标记为最近使用
                return cache[key]
            
            misses += 1
            result = fn(*args, **kwargs)
            cache[key] = result
            
            if len(cache) > maxsize:
                cache.popitem(last=False)  # 删除最久未使用的
            
            return result
        
        def cache_info():
            return {'hits': hits, 'misses': misses, 'size': len(cache), 'maxsize': maxsize}
        
        def cache_clear():
            nonlocal hits, misses
            cache.clear()
            hits = misses = 0
        
        wrapper.cache_info = cache_info
        wrapper.cache_clear = cache_clear
        return wrapper
    return decorator


# 测试: 斐波那契 (经典缓存场景)
print("\n" + "=" * 60)
print("@lru_cache 装饰器 (手写)")
print("=" * 60)

# 无缓存
def fib_no_cache(n):
    if n <= 1: return n
    return fib_no_cache(n-1) + fib_no_cache(n-2)

# 手写缓存
@lru_cache(maxsize=256)
def fib_cached(n):
    if n <= 1: return n
    return fib_cached(n-1) + fib_cached(n-2)

# 官方缓存 (对比)
@functools.lru_cache(maxsize=256)
def fib_official(n):
    if n <= 1: return n
    return fib_official(n-1) + fib_official(n-2)

N = 35

start = time.perf_counter()
r1 = fib_no_cache(N)
t1 = time.perf_counter() - start

start = time.perf_counter()
r2 = fib_cached(N)
t2 = time.perf_counter() - start

start = time.perf_counter()
r3 = fib_official(N)
t3 = time.perf_counter() - start

print(f"\nfib({N}) = {r1}")
print(f"  无缓存:     {t1:.4f}s")
print(f"  手写 cache: {t2:.6f}s  (加速 {t1/t2:,.0f}x)")
print(f"  官方 cache: {t3:.6f}s  (加速 {t1/t3:,.0f}x)")
print(f"\n  手写 cache_info: {fib_cached.cache_info()}")

In [ ]:
# ===================== @validate 参数验证 =====================
def validate(**validators):
    """
    参数验证装饰器
    
    用法:
        @validate(x=lambda v: v > 0, name=lambda v: isinstance(v, str))
        def func(x, name): ...
    """
    def decorator(fn):
        sig = inspect.signature(fn)
        
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            # 绑定参数
            bound = sig.bind(*args, **kwargs)
            bound.apply_defaults()
            
            # 验证
            for param_name, check_fn in validators.items():
                if param_name in bound.arguments:
                    value = bound.arguments[param_name]
                    if not check_fn(value):
                        raise ValueError(
                            f"参数 '{param_name}' 验证失败: "
                            f"值={value}, 规则={check_fn.__doc__ or check_fn}"
                        )
            
            return fn(*args, **kwargs)
        return wrapper
    return decorator


# 测试
print("\n" + "=" * 60)
print("@validate 装饰器")
print("=" * 60)

def positive(v):
    """必须为正数"""
    return isinstance(v, (int, float)) and v > 0

@validate(lr=positive, epochs=positive)
def train(model_name, lr=0.001, epochs=10):
    print(f"  训练 {model_name}: lr={lr}, epochs={epochs}")

train("ResNet", lr=0.01, epochs=50)

try:
    train("ResNet", lr=-0.01)  # 应该失败
except ValueError as e:
    print(f"  ✗ 验证失败: {e}")

In [ ]:
# ===================== 装饰器叠加 =====================
print("\n" + "=" * 60)
print("装饰器叠加 (执行顺序)")
print("=" * 60)

@timer(unit='ms')
@retry(max_attempts=3, delay=0.05, exceptions=(ValueError,))
def risky_computation(n):
    if random.random() < 0.5:
        raise ValueError("随机失败")
    return n ** 2

random.seed(42)
try:
    result = risky_computation(10)
    print(f"  结果: {result}")
except ValueError:
    print("  全部重试失败")

print("\n💡 叠加顺序: 从下往上包装，从上往下执行")
print("   @timer → @retry → func")
print("   等价于: timer(retry(func))")
print("   调用时: timer.wrapper → retry.wrapper → func")

---
## 3. 描述符协议 (Descriptor Protocol) <a id='3'></a>

### 3.1 什么是描述符？

描述符是实现了 `__get__`、`__set__`、`__delete__` 中任一方法的对象。

Python 的 `property`、`classmethod`、`staticmethod`、`__slots__` 底层都是描述符！

| 类型 | 方法 | 例子 |
|------|------|------|
| Data descriptor | `__get__` + `__set__` | property |
| Non-data descriptor | 只有 `__get__` | function, classmethod |

In [ ]:
# 3.1 手写 property (理解描述符)
print("=" * 60)
print("描述符: 手写 property")
print("=" * 60)

class MyProperty:
    """手写 property 描述符"""
    def __init__(self, fget=None, fset=None, fdel=None, doc=None):
        self.fget = fget
        self.fset = fset
        self.fdel = fdel
        self.__doc__ = doc or (fget.__doc__ if fget else None)
    
    def __get__(self, obj, objtype=None):
        if obj is None:
            return self  # 类访问返回描述符本身
        if self.fget is None:
            raise AttributeError("不可读")
        return self.fget(obj)
    
    def __set__(self, obj, value):
        if self.fset is None:
            raise AttributeError("不可写")
        self.fset(obj, value)
    
    def setter(self, fset):
        return type(self)(self.fget, fset, self.fdel, self.__doc__)


class Temperature:
    def __init__(self, celsius):
        self._celsius = celsius
    
    @MyProperty
    def celsius(self):
        return self._celsius
    
    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError("低于绝对零度")
        self._celsius = value
    
    @MyProperty
    def fahrenheit(self):
        return self._celsius * 9/5 + 32

t = Temperature(100)
print(f"  {t.celsius}°C = {t.fahrenheit}°F")
t.celsius = 0
print(f"  {t.celsius}°C = {t.fahrenheit}°F")

try:
    t.celsius = -300
except ValueError as e:
    print(f"  ✗ {e}")

In [ ]:
# 3.2 类型检查描述符 (实用场景)
print("\n" + "=" * 60)
print("类型检查描述符")
print("=" * 60)

class TypeChecked:
    """类型检查描述符: 赋值时自动验证类型"""
    def __init__(self, name, expected_type):
        self.name = name
        self.expected_type = expected_type
        self.storage_name = f'_typechecked_{name}'
    
    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return getattr(obj, self.storage_name, None)
    
    def __set__(self, obj, value):
        if not isinstance(value, self.expected_type):
            raise TypeError(
                f"{self.name} 需要 {self.expected_type.__name__}, "
                f"得到 {type(value).__name__}"
            )
        setattr(obj, self.storage_name, value)


class TrainingConfig:
    lr = TypeChecked('lr', float)
    epochs = TypeChecked('epochs', int)
    model_name = TypeChecked('model_name', str)
    
    def __init__(self, model_name, lr, epochs):
        self.model_name = model_name
        self.lr = lr
        self.epochs = epochs
    
    def __repr__(self):
        return f"Config({self.model_name}, lr={self.lr}, epochs={self.epochs})"


config = TrainingConfig("ResNet50", 0.001, 100)
print(f"  {config}")

try:
    config.lr = "not_a_number"  # 类型错误
except TypeError as e:
    print(f"  ✗ {e}")

try:
    config.epochs = 3.14  # int != float
except TypeError as e:
    print(f"  ✗ {e}")

In [ ]:
# 3.3 描述符查找顺序 (非常重要!)
print("\n" + "=" * 60)
print("属性查找优先级 (MRO + 描述符)")
print("=" * 60)

print("""
Python 属性查找顺序:

  1. Data descriptor (类上定义了 __get__ + __set__)
  2. 实例 __dict__
  3. Non-data descriptor (类上只定义了 __get__)
  4. 类 __dict__ (沿 MRO 查找)
  5. __getattr__ (最后的 fallback)

关键: Data descriptor 的优先级 > 实例 __dict__
      这就是为什么 property 能拦截赋值操作!
""")

# 验证: data descriptor 优先于实例 __dict__
class DataDesc:
    def __get__(self, obj, objtype=None):
        return "from data descriptor"
    def __set__(self, obj, value):
        pass  # 拦截赋值

class NonDataDesc:
    def __get__(self, obj, objtype=None):
        return "from non-data descriptor"

class MyClass:
    data_attr = DataDesc()
    nondata_attr = NonDataDesc()

obj = MyClass()
obj.__dict__['data_attr'] = "from instance dict"
obj.__dict__['nondata_attr'] = "from instance dict"

print(f"  Data descriptor:     {obj.data_attr}  ← 描述符胜出")
print(f"  Non-data descriptor: {obj.nondata_attr}  ← 实例 __dict__ 胜出")

---
## 4. 元类 (Metaclass) <a id='4'></a>

### 4.1 元类是什么？

**类是对象的模板，元类是类的模板。**

```
metaclass → 创建 → class → 创建 → instance
  type          MyClass         obj
```

默认的元类是 `type`。

### 4.2 `__new__` vs `__init__`

| 方法 | 作用 | 时机 |
|------|------|------|
| `__new__` | 创建实例 (分配内存) | 在 `__init__` 之前 |
| `__init__` | 初始化实例 (设置属性) | `__new__` 之后 |

对于元类:
- `Meta.__new__`: 创建类对象 (可以修改类的定义)
- `Meta.__init__`: 初始化类对象

In [ ]:
# 4.1 用 type() 动态创建类
print("=" * 60)
print("用 type() 动态创建类")
print("=" * 60)

# 正常写法
class Dog:
    species = "Canis familiaris"
    def bark(self):
        return "Woof!"

# 等价的 type() 写法
DogDynamic = type(
    'DogDynamic',          # 类名
    (object,),             # 基类元组
    {                      # 类属性和方法字典
        'species': 'Canis familiaris',
        'bark': lambda self: 'Woof!'
    }
)

d1 = Dog()
d2 = DogDynamic()
print(f"  Dog.bark():        {d1.bark()}")
print(f"  DogDynamic.bark(): {d2.bark()}")
print(f"  type(Dog):         {type(Dog)}")
print(f"  type(DogDynamic):  {type(DogDynamic)}")
print("\n💡 class 语句本质上就是调用 type() 来创建类")

In [ ]:
# 4.2 自定义元类
print("\n" + "=" * 60)
print("自定义元类")
print("=" * 60)

class VerboseMeta(type):
    """
    元类: 在创建类时打印信息
    展示 __new__ 和 __init__ 的区别
    """
    def __new__(mcs, name, bases, namespace):
        print(f"  [Meta.__new__] 创建类 '{name}'")
        print(f"    基类: {[b.__name__ for b in bases]}")
        print(f"    属性: {[k for k in namespace if not k.startswith('__')]}")
        
        # 可以在这里修改类定义
        cls = super().__new__(mcs, name, bases, namespace)
        return cls
    
    def __init__(cls, name, bases, namespace):
        print(f"  [Meta.__init__] 初始化类 '{name}'")
        super().__init__(name, bases, namespace)

print("定义 Animal 类:")
class Animal(metaclass=VerboseMeta):
    sound = "..."
    
    def speak(self):
        return self.sound

print("\n定义 Cat 类 (继承 Animal):")
class Cat(Animal):
    sound = "Meow"

print(f"\n  Cat().speak() = {Cat().speak()}")

---
## 5. 元类实战: 自动注册表 <a id='5'></a>

实际应用: 自动收集所有子类 (如插件系统、模型注册)。

这个模式在 PyTorch 和 ML 框架中很常见。

In [ ]:
# 5.1 元类注册表
print("=" * 60)
print("元类实战: 自动注册表")
print("=" * 60)

class RegistryMeta(type):
    """元类: 自动注册所有子类"""
    _registry: Dict[str, type] = {}
    
    def __new__(mcs, name, bases, namespace):
        cls = super().__new__(mcs, name, bases, namespace)
        # 跳过基类本身
        if bases:  # 只注册子类
            registry_name = namespace.get('name', name.lower())
            mcs._registry[registry_name] = cls
            print(f"  [注册] '{registry_name}' → {name}")
        return cls
    
    @classmethod
    def get(mcs, name):
        return mcs._registry.get(name)
    
    @classmethod
    def list_all(mcs):
        return dict(mcs._registry)


class BaseModel(metaclass=RegistryMeta):
    """所有模型的基类"""
    def predict(self, x):
        raise NotImplementedError

class LinearRegression(BaseModel):
    name = 'linear'
    def predict(self, x):
        return f"Linear({x})"

class RandomForest(BaseModel):
    name = 'rf'
    def predict(self, x):
        return f"RF({x})"

class NeuralNetwork(BaseModel):
    name = 'nn'
    def predict(self, x):
        return f"NN({x})"

# 使用注册表
print(f"\n已注册的模型: {RegistryMeta.list_all()}")

# 通过名字获取模型类
model_cls = RegistryMeta.get('rf')
if model_cls:
    model = model_cls()
    print(f"  RegistryMeta.get('rf').predict(42) = {model.predict(42)}")

In [ ]:
# 5.2 更优雅的方式: __init_subclass__ (Python 3.6+, 不需要元类)
print("\n" + "=" * 60)
print("__init_subclass__: 不用元类的注册表")
print("=" * 60)

class PluginBase:
    """使用 __init_subclass__ 实现自动注册"""
    _plugins: Dict[str, type] = {}
    
    def __init_subclass__(cls, plugin_name=None, **kwargs):
        super().__init_subclass__(**kwargs)
        name = plugin_name or cls.__name__.lower()
        PluginBase._plugins[name] = cls
        print(f"  [注册插件] '{name}' → {cls.__name__}")
    
    @classmethod
    def create(cls, name, *args, **kwargs):
        plugin_cls = cls._plugins.get(name)
        if plugin_cls is None:
            raise ValueError(f"未知插件: {name}. 可用: {list(cls._plugins.keys())}")
        return plugin_cls(*args, **kwargs)

class JSONExporter(PluginBase, plugin_name='json'):
    def export(self, data):
        return f"{{json: {data}}}"

class CSVExporter(PluginBase, plugin_name='csv'):
    def export(self, data):
        return f"col1,col2\n{data}"

# 工厂方法创建
exporter = PluginBase.create('json')
print(f"\n  PluginBase.create('json').export('hello') = {exporter.export('hello')}")
print(f"  可用插件: {list(PluginBase._plugins.keys())}")

print("\n💡 __init_subclass__ 是元类的轻量替代，Python 3.6+ 推荐使用")

---
## 6. 设计模式: 工厂模式 (Factory) <a id='6'></a>

**意图**: 将对象创建逻辑与使用逻辑分离。

**ML 场景**: 根据配置字符串创建不同的优化器、损失函数、数据增强策略等。

In [ ]:
print("=" * 60)
print("设计模式: 工厂模式")
print("=" * 60)

# 6.1 简单工厂
class Optimizer(ABC):
    @abstractmethod
    def step(self, params, grads):
        pass
    
    @abstractmethod
    def __repr__(self):
        pass

class SGD(Optimizer):
    def __init__(self, lr=0.01, momentum=0.0):
        self.lr = lr
        self.momentum = momentum
    
    def step(self, params, grads):
        return [p - self.lr * g for p, g in zip(params, grads)]
    
    def __repr__(self):
        return f"SGD(lr={self.lr}, momentum={self.momentum})"

class Adam(Optimizer):
    def __init__(self, lr=0.001, betas=(0.9, 0.999)):
        self.lr = lr
        self.betas = betas
    
    def step(self, params, grads):
        return [p - self.lr * g for p, g in zip(params, grads)]  # 简化
    
    def __repr__(self):
        return f"Adam(lr={self.lr}, betas={self.betas})"


class OptimizerFactory:
    """工厂类: 根据字符串创建优化器"""
    _optimizers = {
        'sgd': SGD,
        'adam': Adam,
    }
    
    @classmethod
    def register(cls, name, optimizer_cls):
        cls._optimizers[name] = optimizer_cls
    
    @classmethod
    def create(cls, name, **kwargs):
        if name not in cls._optimizers:
            raise ValueError(f"未知优化器: {name}. 可选: {list(cls._optimizers.keys())}")
        return cls._optimizers[name](**kwargs)


# 使用
config = {'optimizer': 'adam', 'lr': 0.003}
opt = OptimizerFactory.create(config['optimizer'], lr=config['lr'])
print(f"  从配置创建: {opt}")

# 动态注册新优化器
class AdamW(Adam):
    def __init__(self, lr=0.001, weight_decay=0.01, **kwargs):
        super().__init__(lr=lr, **kwargs)
        self.weight_decay = weight_decay
    def __repr__(self):
        return f"AdamW(lr={self.lr}, wd={self.weight_decay})"

OptimizerFactory.register('adamw', AdamW)
opt2 = OptimizerFactory.create('adamw', lr=0.001, weight_decay=0.05)
print(f"  动态注册后: {opt2}")

---
## 7. 设计模式: 策略模式 (Strategy) <a id='7'></a>

**意图**: 定义一系列算法，使它们可以互换。

**ML 场景**: 不同的数据增强策略、不同的采样策略、不同的评估指标。

In [ ]:
print("\n" + "=" * 60)
print("设计模式: 策略模式")
print("=" * 60)

# 7.1 经典实现 (使用 ABC)
class AugmentationStrategy(ABC):
    @abstractmethod
    def augment(self, data: list) -> list:
        pass

class NoAugmentation(AugmentationStrategy):
    def augment(self, data):
        return data

class RandomFlip(AugmentationStrategy):
    def augment(self, data):
        return [x if random.random() > 0.5 else -x for x in data]

class AddNoise(AugmentationStrategy):
    def __init__(self, scale=0.1):
        self.scale = scale
    
    def augment(self, data):
        return [x + random.gauss(0, self.scale) for x in data]

class ComposedAugmentation(AugmentationStrategy):
    """组合多个策略"""
    def __init__(self, *strategies):
        self.strategies = strategies
    
    def augment(self, data):
        for strategy in self.strategies:
            data = strategy.augment(data)
        return data


class DataPipeline:
    """数据处理管道: 使用策略模式切换增强方式"""
    def __init__(self, strategy: AugmentationStrategy = None):
        self.strategy = strategy or NoAugmentation()
    
    def set_strategy(self, strategy: AugmentationStrategy):
        self.strategy = strategy
    
    def process(self, data):
        augmented = self.strategy.augment(data)
        return augmented


# 使用
data = [1.0, 2.0, 3.0, 4.0, 5.0]
pipeline = DataPipeline()

print(f"  原始数据: {data}")
print(f"  无增强:   {pipeline.process(data)}")

pipeline.set_strategy(RandomFlip())
random.seed(42)
print(f"  随机翻转: {pipeline.process(data)}")

pipeline.set_strategy(AddNoise(scale=0.05))
result = pipeline.process(data)
print(f"  加噪声:   {[f'{x:.3f}' for x in result]}")

# 组合策略
pipeline.set_strategy(ComposedAugmentation(RandomFlip(), AddNoise(0.01)))
result = pipeline.process(data)
print(f"  翻转+噪声: {[f'{x:.3f}' for x in result]}")

In [ ]:
# 7.2 Pythonic 策略模式 (函数即策略)
print("\n--- Pythonic 策略模式 (函数替代类) ---")

def no_aug(data): return data
def flip_aug(data): return [-x for x in data]
def noise_aug(data, scale=0.1): return [x + random.gauss(0, scale) for x in data]

class PipelineV2:
    def __init__(self, aug_fn=None):
        self.aug_fn = aug_fn or no_aug
    
    def process(self, data):
        return self.aug_fn(data)

p = PipelineV2(flip_aug)
print(f"  函数策略: {p.process([1, 2, 3])}")

# lambda 也可以
p2 = PipelineV2(lambda d: [x * 2 for x in d])
print(f"  Lambda 策略: {p2.process([1, 2, 3])}")

print("\n💡 Python 中函数是一等公民，简单策略用函数/lambda 更 Pythonic")

---
## 8. 设计模式: 观察者模式 (Observer) <a id='8'></a>

**意图**: 对象状态变化时自动通知所有关注者。

**ML 场景**: 训练回调 (callback)、日志记录、指标追踪、Early Stopping。

PyTorch Lightning 的 Callback 系统就是观察者模式。

In [ ]:
print("=" * 60)
print("设计模式: 观察者模式 (训练回调系统)")
print("=" * 60)

# 8.1 事件系统
class EventEmitter:
    """通用事件发射器"""
    def __init__(self):
        self._handlers: Dict[str, List[Callable]] = defaultdict(list)
    
    def on(self, event: str, handler: Callable):
        """注册事件处理器"""
        self._handlers[event].append(handler)
        return self  # 链式调用
    
    def off(self, event: str, handler: Callable):
        """注销事件处理器"""
        self._handlers[event].remove(handler)
    
    def emit(self, event: str, **kwargs):
        """触发事件"""
        for handler in self._handlers.get(event, []):
            handler(**kwargs)


# 8.2 训练回调
class Callback:
    """回调基类"""
    def on_train_begin(self, **kwargs): pass
    def on_epoch_begin(self, **kwargs): pass
    def on_batch_end(self, **kwargs): pass
    def on_epoch_end(self, **kwargs): pass
    def on_train_end(self, **kwargs): pass


class PrintCallback(Callback):
    """打印训练进度"""
    def on_epoch_end(self, epoch, loss, **kwargs):
        print(f"    [Print] Epoch {epoch}: loss={loss:.4f}")


class HistoryCallback(Callback):
    """记录训练历史"""
    def __init__(self):
        self.history = {'loss': [], 'epoch': []}
    
    def on_epoch_end(self, epoch, loss, **kwargs):
        self.history['loss'].append(loss)
        self.history['epoch'].append(epoch)


class EarlyStopping(Callback):
    """早停回调"""
    def __init__(self, patience=3, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float('inf')
        self.counter = 0
        self.should_stop = False
    
    def on_epoch_end(self, epoch, loss, **kwargs):
        if loss < self.best_loss - self.min_delta:
            self.best_loss = loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
                print(f"    [EarlyStopping] 停止! (patience={self.patience})")


class LRScheduler(Callback):
    """学习率调度回调"""
    def __init__(self, initial_lr=0.01, decay=0.95):
        self.lr = initial_lr
        self.decay = decay
    
    def on_epoch_end(self, epoch, **kwargs):
        self.lr *= self.decay
        print(f"    [LR] lr → {self.lr:.6f}")

In [ ]:
# 8.3 简易训练器 (使用回调)
class SimpleTrainer:
    """使用观察者模式的训练器"""
    def __init__(self, callbacks: List[Callback] = None):
        self.callbacks = callbacks or []
    
    def _notify(self, event_name, **kwargs):
        for cb in self.callbacks:
            method = getattr(cb, event_name, None)
            if method:
                method(**kwargs)
    
    def train(self, n_epochs=10):
        self._notify('on_train_begin', n_epochs=n_epochs)
        
        for epoch in range(1, n_epochs + 1):
            self._notify('on_epoch_begin', epoch=epoch)
            
            # 模拟训练 (loss 逐渐下降然后趋于平稳)
            loss = 1.0 / (1 + epoch * 0.3) + random.gauss(0, 0.02)
            
            self._notify('on_epoch_end', epoch=epoch, loss=loss)
            
            # 检查早停
            for cb in self.callbacks:
                if isinstance(cb, EarlyStopping) and cb.should_stop:
                    self._notify('on_train_end', epoch=epoch)
                    return
        
        self._notify('on_train_end', epoch=n_epochs)


# 使用
print("\n训练 (带回调):")
history = HistoryCallback()
trainer = SimpleTrainer(callbacks=[
    PrintCallback(),
    history,
    EarlyStopping(patience=5, min_delta=0.01),
    LRScheduler(initial_lr=0.01, decay=0.9),
])

random.seed(42)
trainer.train(n_epochs=20)

In [ ]:
# 可视化训练历史
import matplotlib.pyplot as plt

if history.history['loss']:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(history.history['epoch'], history.history['loss'], 'b-o', linewidth=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('训练 Loss (由 HistoryCallback 记录)')
    plt.tight_layout()
    plt.show()

---
## 9. 综合实战: ML 训练框架中的设计模式 <a id='9'></a>

综合运用所有设计模式，构建一个迷你 ML 框架。

In [ ]:
# ===================== 综合实战 =====================
print("=" * 60)
print("综合实战: 迷你 ML 框架")
print("=" * 60)

# --- 注册表 (元类 / __init_subclass__) ---
class ModelRegistry:
    _models = {}
    
    @classmethod
    def register(cls, name):
        """装饰器: 注册模型类"""
        def decorator(model_cls):
            cls._models[name] = model_cls
            return model_cls
        return decorator
    
    @classmethod
    def create(cls, name, **kwargs):
        return cls._models[name](**kwargs)
    
    @classmethod
    def available(cls):
        return list(cls._models.keys())

class LossRegistry:
    _losses = {}
    
    @classmethod
    def register(cls, name):
        def decorator(loss_cls):
            cls._losses[name] = loss_cls
            return loss_cls
        return decorator
    
    @classmethod
    def create(cls, name, **kwargs):
        return cls._losses[name](**kwargs)


# --- 模型定义 (工厂模式 + 注册表) ---
@ModelRegistry.register('linear')
class LinearModel:
    def __init__(self, input_dim=10, output_dim=1):
        self.w = [random.gauss(0, 0.01) for _ in range(input_dim)]
        self.b = 0.0
        self.name = f"Linear({input_dim}→{output_dim})"
    
    def forward(self, x):
        return sum(wi * xi for wi, xi in zip(self.w, x)) + self.b

@ModelRegistry.register('mlp')
class MLPModel:
    def __init__(self, input_dim=10, hidden_dim=32, output_dim=1):
        self.name = f"MLP({input_dim}→{hidden_dim}→{output_dim})"
    
    def forward(self, x):
        return sum(x) * 0.1  # 简化


# --- 损失函数 (策略模式 + 注册表) ---
@LossRegistry.register('mse')
class MSELoss:
    def __call__(self, pred, target):
        return (pred - target) ** 2

@LossRegistry.register('mae')
class MAELoss:
    def __call__(self, pred, target):
        return abs(pred - target)


# --- 从配置构建 ---
config = {
    'model': {'name': 'mlp', 'input_dim': 5, 'hidden_dim': 16},
    'loss': 'mse',
    'epochs': 5,
}

model = ModelRegistry.create(config['model']['name'], **{k: v for k, v in config['model'].items() if k != 'name'})
loss_fn = LossRegistry.create(config['loss'])

print(f"\n  可用模型: {ModelRegistry.available()}")
print(f"  创建模型: {model.name}")
print(f"  损失函数: {config['loss']}")

# 模拟训练
x = [1.0, 2.0, 3.0, 4.0, 5.0]
target = 3.0
pred = model.forward(x)
loss = loss_fn(pred, target)
print(f"\n  前向: pred={pred:.4f}, target={target}, loss={loss:.4f}")

print("\n💡 这就是 PyTorch Lightning / HuggingFace Trainer 的设计思想:")
print("   注册表 (模型/损失/优化器) + 工厂 (从配置创建) + 回调 (观察者)")

---
## 10. 思考题 & 总结 <a id='10'></a>

### 今日核心知识点

1. **装饰器三层**: 无参 / 带参 / 可选参数；`@functools.wraps` 保留元信息
2. **描述符**: `__get__` / `__set__` → property 的本质；Data vs Non-data 优先级
3. **元类**: `type()` 创建类；自定义元类 vs `__init_subclass__`
4. **工厂模式**: 注册表 + 从配置/字符串创建对象
5. **策略模式**: 可互换的算法；Python 用函数更 Pythonic
6. **观察者模式**: 回调系统；事件驱动；ML 训练的标配

### 面试高频问题

1. **装饰器的原理？`@wraps` 的作用？**
2. **`property` 的底层实现？** → 描述符
3. **`__new__` vs `__init__`？** → 创建 vs 初始化
4. **如何实现单例模式？** → 装饰器 / 元类 / `__new__` / 模块
5. **Python 中设计模式和其他语言有什么不同？** → 函数是一等公民，很多模式可以简化

### 实际工程中的应用

| 模式 | 实际框架 |
|------|----------|
| 装饰器 | Flask @route, PyTest @fixture |
| 描述符 | Django ORM fields, SQLAlchemy columns |
| 注册表 | MMCV, Detectron2 模型注册 |
| 工厂 | HuggingFace AutoModel.from_pretrained() |
| 策略 | torchvision.transforms |
| 观察者 | PyTorch Lightning Callbacks, Keras |

### 明日预习: PyTorch Tensor + Autograd
- Tensor 的 Storage / Stride / View
- 计算图 DAG
- 反向传播链式法则

In [ ]:
print("\n" + "=" * 60)
print("W1 Day2 完成! 🎉")
print("=" * 60)
print("""
今日成果:
  ✅ 装饰器三层结构 (无参 / 带参 / 可选参数 / 类装饰器)
  ✅ 装饰器库: @timer @retry @lru_cache @validate
  ✅ 描述符协议: 手写 property + 类型检查描述符 + 查找优先级
  ✅ 元类: type() 动态创建 + 自定义元类 + __init_subclass__
  ✅ 元类实战: 自动注册表
  ✅ 工厂模式: 注册表 + 从配置创建
  ✅ 策略模式: ABC 版 + Pythonic 函数版
  ✅ 观察者模式: 训练回调系统 (PrintCB, History, EarlyStopping, LRScheduler)
  ✅ 综合实战: 迷你 ML 框架 (注册表 + 工厂 + 策略 + 回调)
""")